In [ ]:
!pip install rfdetr[onnx]

In [26]:
from rfdetr import RFDETRBase, RFDETRMedium, RFDETRNano
import cv2
import numpy as np
import onnxruntime as ort
import time
from tqdm.notebook import tqdm

# RFDETR

In [ ]:
model = RFDETRMedium()
model.export()
!cp output/inference_model.onnx rfdetr_medium.onnx

In [ ]:
model = RFDETRBase()
model.export()
!cp output/inference_model.onnx rfdetr_base.onnx

In [ ]:
model = RFDETRNano()
model.export()
!cp output/inference_model.onnx rfdetr_nano.onnx

In [40]:
IMG_SIZE = 560
CONF_THRESHOLD = 0.4
ONNX_PATH = "rfdetr_base.onnx"


providers = ['CUDAExecutionProvider']
#providers = ["CPUExecutionProvider"]
session = ort.InferenceSession(ONNX_PATH, providers=providers)

model_input = session.get_inputs()[0]
input_name = model_input.name
input_shape = model_input.shape

IMG_H = int(input_shape[2])
IMG_W = int(input_shape[3])

print(f"Model Loaded: {ONNX_PATH}")
print(f"Detected Input Name: {input_name}")
print(f"Detected Input Shape: {IMG_W}x{IMG_H}")

Model Loaded: rfdetr_base.onnx
Detected Input Name: input
Detected Input Shape: 560x560


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [41]:
def preprocess(frame, target_w, target_h):
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (target_w, target_h))

    img = img.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std

    img = img.transpose(2, 0, 1)[np.newaxis, ...]
    return np.ascontiguousarray(img).astype(np.float32)

def postprocess(outputs, orig_shape, threshold=0.4):
    if outputs[0].shape[-1] == 4:
        pred_boxes, pred_logits = outputs[0], outputs[1]
    else:
        pred_logits, pred_boxes = outputs[0], outputs[1]

    h_orig, w_orig = orig_shape[:2]
    probs = sigmoid(pred_logits[0])
    scores = np.max(probs, axis=-1)
    class_ids = np.argmax(probs, axis=-1)

    mask = scores > threshold
    valid_boxes = pred_boxes[0][mask]
    valid_scores = scores[mask]
    valid_labels = class_ids[mask]

    final_results = []
    for i in range(len(valid_boxes)):
        cx, cy, w, h = valid_boxes[i]

        # RF-DETR coordinates are normalized (0 to 1)
        # scale them directly to the original video resolution
        x1 = (cx - w / 2) * w_orig
        y1 = (cy - h / 2) * h_orig
        x2 = (cx + w / 2) * w_orig
        y2 = (cy + h / 2) * h_orig

        final_results.append({
            "box": [int(x1), int(y1), int(x2), int(y2)],
            "score": float(valid_scores[i]),
            "class_id": int(valid_labels[i])
        })
    return final_results

def draw_info(frame, detections):
    """Draws boxes and labels on the frame."""
    for det in detections:
        x1, y1, x2, y2 = det["box"]
        label = f"ID:{det['class_id']} {det['score']:.2f}"

        # Draw Box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Draw Label Background
        cv2.rectangle(frame, (x1, y1 - 20), (x1 + 100, y1), (0, 255, 0), -1)
        cv2.putText(frame, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return frame

In [ ]:
input_path = "people_walking.mp4"
output_path = "detr.mp4"


cap = cv2.VideoCapture(input_path)
w_vid, h_vid = int(cap.get(3)), int(cap.get(4))
fps, total = cap.get(5), int(cap.get(7))

out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w_vid, h_vid))
pbar = tqdm(total=total, desc="🚀 Inferencing")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    blob = preprocess(frame, IMG_W, IMG_H)
    outputs = session.run(None, {input_name: blob})
    detections = postprocess(outputs, (h_vid, w_vid), threshold=0.4)

    frame = draw_info(frame, detections)
    out.write(frame)
    pbar.update(1)

pbar.close()
cap.release()
out.release()
print(f"Success! Output at: {output_path}")